In [1]:
import math
from Utils.hdfs import init
from Utils.dataFrameSize import dfSizeEstimator
from pyspark.sql import SparkSession, Window, types as T, functions as F
from pyspark.storagelevel import StorageLevel

In [2]:
builder = (
    SparkSession.builder
    .appName("create_dims")
    .master("yarn")
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
)
spark = builder.getOrCreate()
init(spark)

In [3]:
%%sql
use warehouse;

""


In [4]:
%%sql
show tables;

,namespace,tableName,isTemporary
0,warehouse,flight_raw,False
1,warehouse,flight,False
2,warehouse,dim_airport,False


In [5]:
%%sql
desc flight;

,col_name,data_type,comment
0,departure_time,string,None
1,arrival_time,timestamp,None
2,flight_id,string,None
3,aircraft_id,string,None
4,itinerary_no,int,None
5,ticket_no,string,None
6,flight_cost,"decimal(10,0)",None
7,origin_airport,string,None
8,destination_airport,string,None
9,frequent_flier,boolean,None


In [7]:
%%sql

    SELECT aircraft_id, MAX(STRUCT(cnt, airport)).airport AS airport
    FROM (
        SELECT aircraft_id, origin_airport AS airport, COUNT(*) AS cnt
        FROM flight
        GROUP BY aircraft_id, origin_airport
    
        UNION ALL
    
        SELECT aircraft_id, destination_airport AS airport, COUNT(*) AS cnt
        FROM flight
        GROUP BY aircraft_id, destination_airport
    ) t
    GROUP BY aircraft_id;

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '('.(line 2, pos 42)

== SQL ==

    select aircraft_id, max(airport) keep (dense_rank first order by cnt desc) as airport
------------------------------------------^^^
    from (
        select 
            aircraft_id, 
            origin_airport as airport, count(*) as cnt
        from flight
        group by aircraft_id, origin_airport
        union all
        select 
            aircraft_id, 
            destination_airport as airport, count(*) as cnt
        from flight
        group by aircraft_id, destination_airport) limit 3;


In [6]:
%%sql
select 
    aircraft_id, 
    flight_cost,
    origin_airport,
    destination_airport, 
    airplane_model, 
    distance, 
    fuel_consumed_litre, 
    avg_flight_speed_kmps, 
    engine_performance
from flight limit 3;

,aircraft_id,flight_cost,origin_airport,destination_airport,airplane_model,distance,fuel_consumed_litre,avg_flight_speed_kmps,engine_performance
0,AI578,6588,Agawam,Fridley,Boeing 747-400F,2074,19043.500000,948.8,17821
1,AI835,5609,Hot Springs,Tahlequah,Boeing 737 MAX 9,2445,18738.199219,942.7,18192
2,AI901,636,Saginaw,Aberdeen,Antonov An-158,3559,68580.296875,949.6,25250


In [11]:
%%sql
with possible_pr as (
    SELECT aircraft_id, max(struct(cnt, airport)).airport AS airport
    FROM (
        SELECT aircraft_id, origin_airport AS airport, COUNT(*) AS cnt
        FROM flight
        GROUP BY aircraft_id, origin_airport
    
        UNION ALL
    
        SELECT aircraft_id, destination_airport AS airport, COUNT(*) AS cnt
        FROM flight
        GROUP BY aircraft_id, destination_airport
    ) t
    GROUP BY aircraft_id)                              
select 
    f.aircraft_id, 
    pp.airport,
    f.airplane_model,
    avg(f.flight_cost) avg_flight_cost,
    sum(f.distance) total_distance, 
    avg(f.fuel_consumed_litre) avg_fuel_consumption, 
    avg(f.avg_flight_speed_kmps) avg_flight_speed_kmps, 
    max(f.engine_performance) as max_engine_performance, 
    min(f.engine_performance) as min_engine_performance
from flight f
join possible_pr pp on pp.aircraft_id = f.aircraft_id
group by 
    f.aircraft_id, 
    pp.airport,
    f.airplane_model
limit 3;

,aircraft_id,airport,airplane_model,avg(flight_cost),total_distance,avg_fuel_consumption,avg_flight_speed_kmps,max_engine_performance,min_engine_performance
0,AI578,Tyler,Boeing 747-400F,4952.2222,35694,48407.988281,1088.80,53457,7173
1,AI173,South Holland,De Havilland Canada DHC-8-300 Dash 8,3917.8750,37170,53028.599854,1212.20,54983,16243
2,AI766,Maple Valley,Douglas DC-9-20,4150.9000,33327,43413.169727,1011.11,41208,5675


In [18]:
%%sql
show tables;

,namespace,tableName,isTemporary
0,warehouse,flight_raw,False
1,warehouse,flight,False
2,warehouse,dim_airport,False
3,warehouse,dim_passenger,False
4,warehouse,scd_dim_aircraft,False


In [20]:
%%sql
select * from scd_dim_aircraft limit 20;

,aircraft_id,airport,airplane_model,avg_flight_cost,total_distance,avg_fuel_consumption,avg_flight_speed_kmps,max_engine_performance,min_engine_performance,dbt_scd_id,dbt_updated_at,dbt_valid_from,dbt_valid_to
0,AI017,Layton,Cessna 206 Stationair,4853.4615,47071,38538.561448,1063.438462,58585,7044,6888fed95b1af224d6c4b7a3bbb4183e,2026-07-05 21:44:30.504326,2026-07-05 21:44:30.504326,NaT
1,AI547,West Linn,Boeing 737-200,4834.0000,33206,35189.090776,968.200000,58964,6432,6e97b2f824cdffdc9354f48f620ea133,2026-07-05 21:44:30.504326,2026-07-05 21:44:30.504326,NaT
2,AI884,Wooster,Pilatus Britten-Norman BN-2A Mk III Trislander,3538.3750,38033,63697.462402,1152.800000,42238,26137,c55240b1e8f454e1c662c19a5349cb6e,2026-07-05 21:44:30.504326,2026-07-05 21:44:30.504326,NaT
3,AI884,Wooster,Antonov AN-72,4046.4444,55593,39429.699897,961.244444,58656,5187,c55240b1e8f454e1c662c19a5349cb6e,2026-07-05 21:44:30.504326,2026-07-05 21:44:30.504326,NaT
4,AI500,Sumter,Airbus A300C4,4052.0000,45629,46675.600446,977.085714,52905,6957,d18a55cb9322bbcba80f068ae4b66440,2026-07-05 21:44:30.504326,2026-07-05 21:44:30.504326,NaT
5,AI500,Sumter,McDonnell Douglas MD-88,5234.3571,35891,33336.964181,892.100000,40161,5334,d18a55cb9322bbcba80f068ae4b66440,2026-07-05 21:44:30.504326,2026-07-05 21:44:30.504326,NaT
6,AI500,Sumter,Douglas DC-6,3352.5000,38587,40596.550334,950.275000,47940,7458,d18a55cb9322bbcba80f068ae4b66440,2026-07-05 21:44:30.504326,2026-07-05 21:44:30.504326,NaT
7,AI493,Peabody,Ilyushin IL62,4457.7778,27467,41469.155490,941.077778,43449,5798,38c8823ab11f7443cd2e67f19c1ec689,2026-07-05 21:44:30.504326,2026-07-05 21:44:30.504326,NaT
8,AI884,Wooster,Piper PA-31 Navajo,2441.7143,48592,46807.256487,1008.307143,51819,6320,c55240b1e8f454e1c662c19a5349cb6e,2026-07-05 21:44:30.504326,2026-07-05 21:44:30.504326,NaT
9,AI884,Wooster,Learjet 35,5769.0000,15861,34074.200195,900.614286,29892,10520,c55240b1e8f454e1c662c19a5349cb6e,2026-07-05 21:44:30.504326,2026-07-05 21:44:30.504326,NaT
